# hypercomplex

> Parameterized hypercomplex layers over the views axis, and a first
> representation-learning model on the `[views, channels, days]` array.

`01_data` produces several orderings of the same exchange; `02_scattering`
stacks them as **views**, `[n, C, ...]`. This notebook treats the `n` views as
the components of an `n`-dimensional hypercomplex number and learns how they
should mix, using **parameterized hypercomplex (PH) layers** (Zhang et al.
2021, *Beyond Fully-Connected Layers with Quaternions*; Grassucci et al. 2022,
*PHNNs: Lightweight Neural Networks via Parameterized Hypercomplex
Convolutions*).

The PH idea in one line: a weight matrix is built as a sum of Kronecker products

$$W = \sum_{i=1}^{n} A_i \otimes S_i,$$

where the small $A_i \in \mathbb{R}^{n\times n}$ are **learned** and encode the
algebra (for $n=4$ with the Hamilton matrices fixed, this is exactly a
quaternion layer), and the $S_i$ hold the ordinary filter weights. Parameters
drop by roughly $1/n$ compared to a real layer of the same width, and -- the
interesting part for us -- the learned $A_i$ can be read afterwards as *how
view $i$ modulates the interaction between all the other views*. Financial
data has no algebra to enforce, so this is PHNN as a **structure-discovery**
tool rather than a symmetry prior.

**Reimplemented, not vendored.** The layers below follow Eleonora Grassucci's
reference implementation (`HyperNets`, MIT); the notice is reproduced in the
next cell so it ships with the module. Everything else is new and, as in
`01_data` / `02_scattering`, device-agnostic and tested on synthetic data.

In [ ]:
#| default_exp hypercomplex

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import matplotlib
matplotlib.use("Agg")
%matplotlib inline

In [ ]:
#| export
# The PH layers in this module are a re-implementation following
# E. Grassucci et al., "HyperNets: Hypercomplex Neural Networks with PyTorch",
# https://github.com/eleGAN23/HyperNets -- released under the MIT License:
GRASSUCCI_LICENSE = """
MIT License

Copyright (c) 2021 Eleonora Grassucci

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## The Kronecker sum

One function builds the full weight from `(A, S)` for linear *and* conv
layers: `S` may carry trailing kernel dims, which pass through untouched.
Written as a single `einsum` + `reshape` rather than a loop over `torch.kron`,
so it is one fused op on whatever device the parameters live on.

In [ ]:
#| export
def ph_weight(A, S):
    """`A: [n, n, n]`, `S: [n, out/n, in/n, *kernel]` -> `W: [out, in, *kernel]`
    with `W = sum_i kron(A[i], S[i])` (kernel dims ride along)."""
    n, o, i = S.shape[:3]
    W = torch.einsum("nab,ncd...->acbd...", A, S)      # [n, o, n, i, *k]
    return W.reshape(n * o, n * i, *S.shape[3:])


def hamilton_basis():
    """The four `[4, 4]` matrices of left-multiplication by `1, i, j, k`. With
    `A = hamilton_basis()` a PH layer *is* a quaternion layer."""
    return torch.tensor([
        [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]],
        [[0, -1, 0, 0], [1, 0, 0, 0], [0, 0, 0, -1], [0, 0, 1, 0]],
        [[0, 0, -1, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, -1, 0, 0]],
        [[0, 0, 0, -1], [0, 0, -1, 0], [0, 1, 0, 0], [1, 0, 0, 0]],
    ], dtype=torch.float32)

In [ ]:
torch.manual_seed(0)
# matches an explicit loop of torch.kron, linear and conv cases
A, S = torch.randn(3, 3, 3), torch.randn(3, 2, 4)
W = ph_weight(A, S)
assert W.shape == (6, 12)
assert torch.allclose(W, sum(torch.kron(A[i], S[i]) for i in range(3)), atol=1e-6)
Sk = torch.randn(3, 2, 4, 5)
Wk = ph_weight(A, Sk)
assert Wk.shape == (6, 12, 5)
assert torch.allclose(Wk[..., 2], sum(torch.kron(A[i], Sk[i, ..., 2]) for i in range(3)), atol=1e-6)

# with the Hamilton basis, W is left-multiplication by the quaternion q
def qmul(q, p):
    a, b, c, d = q; p0, p1, p2, p3 = p
    return torch.stack([a*p0 - b*p1 - c*p2 - d*p3, a*p1 + b*p0 + c*p3 - d*p2,
                        a*p2 - b*p3 + c*p0 + d*p1, a*p3 + b*p2 - c*p1 + d*p0])
q, p = torch.randn(4), torch.randn(4)
Wq = ph_weight(hamilton_basis(), q.view(4, 1, 1))
assert torch.allclose(Wq @ p, qmul(q, p), atol=1e-6)

## Layers

`PHMLinear` and `PHConv1d` mirror `nn.Linear` / `nn.Conv1d`, with `n` in
front. `in_features` and `out_features` must be multiples of `n` -- the first
axis of the data is read as `n` blocks of `in/n` features, i.e. **view-major**:
`[n * f]` means view 0's `f` features, then view 1's, and so on. That is what
`stack_views(...).reshape(...)` produces, so the two notebooks agree by
construction.

Init: `S` uses Kaiming-uniform like `nn.Linear`, computed for the fan-in of
the *assembled* `W`; `A` is Xavier-uniform as in the reference implementation.
Nothing is done to `A` beyond that -- no orthogonality, no normalization -- so
the algebra is genuinely free to be whatever the data prefers. Constraints
worth trying later (orthogonal `A_i`, sparsity) are a one-line penalty on
`model.A` in the training loop.

In [ ]:
#| export
class PHMLinear(nn.Module):
    """Parameterized hypercomplex linear layer: `W = sum_i kron(A_i, S_i)`."""
    def __init__(self, n, in_features, out_features, bias=True):
        super().__init__()
        assert in_features % n == 0 and out_features % n == 0, "features must be multiples of n"
        self.n, self.in_features, self.out_features = n, in_features, out_features
        self.A = nn.Parameter(nn.init.xavier_uniform_(torch.empty(n, n, n)))
        self.S = nn.Parameter(torch.empty(n, out_features // n, in_features // n))
        self.bias = nn.Parameter(torch.empty(out_features)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1 / math.sqrt(self.in_features)
        nn.init.uniform_(self.S, -bound, bound)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -bound, bound)

    @property
    def weight(self):
        return ph_weight(self.A, self.S)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias)

    def extra_repr(self):
        return f"n={self.n}, in_features={self.in_features}, out_features={self.out_features}"


class PHConv1d(nn.Module):
    """Parameterized hypercomplex 1-D convolution, same signature as
    `nn.Conv1d` with `n` in front."""
    def __init__(self, n, in_channels, out_channels, kernel_size, stride=1, padding=0,
                 dilation=1, bias=True):
        super().__init__()
        assert in_channels % n == 0 and out_channels % n == 0, "channels must be multiples of n"
        self.n, self.in_channels, self.out_channels = n, in_channels, out_channels
        self.kernel_size, self.stride, self.padding, self.dilation = kernel_size, stride, padding, dilation
        self.A = nn.Parameter(nn.init.xavier_uniform_(torch.empty(n, n, n)))
        self.S = nn.Parameter(torch.empty(n, out_channels // n, in_channels // n, kernel_size))
        self.bias = nn.Parameter(torch.empty(out_channels)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1 / math.sqrt(self.in_channels * self.kernel_size)
        nn.init.uniform_(self.S, -bound, bound)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -bound, bound)

    @property
    def weight(self):
        return ph_weight(self.A, self.S)

    def forward(self, x):
        return F.conv1d(x, self.weight, self.bias, stride=self.stride, padding=self.padding,
                        dilation=self.dilation)

    def extra_repr(self):
        return (f"n={self.n}, {self.in_channels}, {self.out_channels}, kernel_size={self.kernel_size}, "
                f"stride={self.stride}, padding={self.padding}")


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
torch.manual_seed(1)
lin = PHMLinear(4, 8, 12)
x = torch.randn(5, 8)
assert lin(x).shape == (5, 12)
assert count_params(lin) == 4 ** 3 + 4 * 3 * 2 + 12
# quaternion special case: freeze A to Hamilton and check against nn.Linear on the assembled W
with torch.no_grad():
    lin.A.copy_(hamilton_basis())
ref = nn.Linear(8, 12); ref.weight.data = lin.weight.detach().clone(); ref.bias.data = lin.bias.detach().clone()
assert torch.allclose(lin(x), ref(x), atol=1e-6)

conv = PHConv1d(2, 4, 6, kernel_size=3, padding=1)
y = conv(torch.randn(3, 4, 20))
assert y.shape == (3, 6, 20)
assert conv(torch.randn(3, 4, 20)).shape == (3, 6, 20)
strided = PHConv1d(2, 4, 6, kernel_size=3, stride=2, padding=1)
assert strided(torch.randn(3, 4, 20)).shape == (3, 6, 10)
# gradients reach both the algebra and the filters
y.sum().backward()
assert conv.A.grad is not None and conv.S.grad is not None and conv.A.grad.abs().sum() > 0
# fewer params than the real conv of the same shape (n^3 overhead aside)
assert count_params(conv) < count_params(nn.Conv1d(4, 6, 3))

## A first model: hypercomplex autoencoder along the channel axis

The easiest thing that could possibly teach us something. Each **day** is a
sample: a `[n, f, C]` slab -- `n` views, `f` features per view (1 for raw
log-prices, `n_paths` for scattering), `C` positions along the ordered channel
axis. A small conv autoencoder runs **along `C`** (the axis the orderings made
smooth, which is the whole point of `01_data`), with PH convolutions doing
the view mixing at every layer. Bottleneck: stride-2 downsampling plus a
narrow `z` code per position.

`make_ae(hyper=False)` builds the identical architecture with plain
`nn.Conv1d`, for a baseline with roughly `n`× the parameters. If PH matches
it at a fraction of the size, the Kronecker structure is a good prior for
"how the views relate"; if it doesn't, that's also an answer.

In [ ]:
#| export
class ConvAE1d(nn.Module):
    """Strided conv encoder / upsampling decoder along the last axis.
    `conv(in, out, k, **kw)` is the layer factory, so PH and real versions
    share one definition."""
    def __init__(self, conv, f_in, h, z, k=5):
        super().__init__()
        p = k // 2
        self.enc = nn.Sequential(
            conv(f_in, h, k, padding=p), nn.GELU(),
            conv(h, h, k, stride=2, padding=p), nn.GELU(),
            conv(h, z, 1))
        self.dec = nn.Sequential(
            conv(z, h, 1), nn.GELU(),
            conv(h, h, k, padding=p), nn.GELU(),
            conv(h, f_in, k, padding=p))

    def encode(self, x):
        return self.enc(x)

    def decode(self, z, length):
        return self.dec(F.interpolate(z, size=length, mode="nearest"))

    def forward(self, x):
        return self.decode(self.encode(x), x.shape[-1])


def make_ae(n, f, h, z, k=5, hyper=True):
    """`[B, n*f, C]` autoencoder. `hyper=True`: every conv is a `PHConv1d`
    with `n` views; `False`: plain `nn.Conv1d` at the same widths (the
    baseline). Widths `h`, `z` are per view, so the tensors are `n*h`, `n*z`."""
    if hyper:
        conv = lambda i, o, k, **kw: PHConv1d(n, i, o, k, **kw)
    else:
        conv = lambda i, o, k, **kw: nn.Conv1d(i, o, k, **kw)
    return ConvAE1d(conv, n * f, n * h, n * z, k)


def ph_algebras(model):
    """All learned `A` tensors in a model, in module order: `[n_layers, n, n, n]`."""
    return torch.stack([m.A.detach() for m in model.modules() if isinstance(m, (PHConv1d, PHMLinear))])

In [ ]:
torch.manual_seed(2)
n, f, C = 4, 3, 50
x = torch.randn(7, n * f, C)
ae = make_ae(n, f, h=8, z=2, hyper=True)
assert ae(x).shape == x.shape
assert ae.encode(x).shape == (7, n * 2, C // 2)
assert ae(torch.randn(7, n * f, C + 1)).shape == (7, n * f, C + 1), "odd lengths round-trip too"
base = make_ae(n, f, h=8, z=2, hyper=False)
assert base(x).shape == x.shape
assert count_params(ae) < count_params(base), (count_params(ae), count_params(base))
assert ph_algebras(ae).shape == (6, n, n, n)

## Data plumbing and training

`views_to_batch` is the one place the axis convention is decided: the
`[n, C, ...]` array from `stack_views` becomes `[T, n*f, C]`, days as the
batch, view-major features, channel position as the conv axis.

`fit_ae` is a bare training loop -- MSE, AdamW, a **time-based** split (last
`val_frac` of days held out, never a random one: adjacent days are near
duplicates, so a random split would leak) -- returning the loss history. No
early stopping or schedulers; this is EDA, and a printed curve is the output.

In [ ]:
#| export
def views_to_batch(V):
    """`[n, C, T]` -> `[T, n, C]`; `[n, C, P, T]` -> `[T, n*P, C]` (view-major
    features). Days become the batch axis, channel position the conv axis."""
    if V.ndim == 3:
        return V.permute(2, 0, 1).contiguous()
    n, C, P, T = V.shape
    return V.permute(3, 0, 2, 1).reshape(T, n * P, C).contiguous()


def fit_ae(model, X, epochs=30, lr=2e-3, batch_size=64, val_frac=0.2, weight_decay=1e-4,
           verbose=True, seed=0):
    """Train `model` to reconstruct `X` (`[T, F, C]`), last `val_frac` of the
    time axis held out. Returns `{"train": [...], "val": [...]}` per epoch."""
    device = X.device
    model = model.to(device)
    T = X.shape[0]
    n_val = max(1, int(round(val_frac * T)))
    Xtr, Xva = X[:T - n_val], X[T - n_val:]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    g = torch.Generator().manual_seed(seed)
    hist = {"train": [], "val": []}
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(Xtr.shape[0], generator=g)
        tot, cnt = 0.0, 0
        for i in range(0, len(perm), batch_size):
            xb = Xtr[perm[i:i + batch_size]]
            loss = F.mse_loss(model(xb), xb)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.item() * xb.shape[0]; cnt += xb.shape[0]
        model.eval()
        with torch.no_grad():
            val = float(np.mean([F.mse_loss(model(Xva[i:i + batch_size]), Xva[i:i + batch_size]).item()
                                 for i in range(0, Xva.shape[0], batch_size)]))
        hist["train"].append(tot / cnt); hist["val"].append(val)
        if verbose and (ep % max(1, epochs // 10) == 0 or ep == epochs - 1):
            print(f"epoch {ep:3d}  train {tot / cnt:.4f}  val {val:.4f}")
    return hist

In [ ]:
torch.manual_seed(3)
V = torch.randn(4, 10, 30)
X = views_to_batch(V)
assert X.shape == (30, 4, 10) and torch.equal(X[5, 2], V[2, :, 5])
V4 = torch.randn(4, 10, 3, 30)
X4 = views_to_batch(V4)
assert X4.shape == (30, 12, 10) and torch.equal(X4[5, 1 * 3 + 2], V4[1, :, 2, 5]), "view-major"

# learns something on smooth synthetic data: loss drops and val is finite
t = torch.linspace(0, 6.28, 40)
base = torch.stack([torch.sin(t + ph) for ph in torch.rand(120) * 6.28])       # [T=120, C=40]
V = torch.stack([base, base.roll(3, -1), -base, base ** 2]).permute(0, 2, 1)     # [n=4, C, T]
X = views_to_batch(V)
ae = make_ae(4, 1, h=8, z=2)
hist = fit_ae(ae, X, epochs=25, lr=5e-3, batch_size=32, verbose=False)
assert hist["train"][-1] < 0.5 * hist["train"][0], hist["train"][::5]
assert np.isfinite(hist["val"][-1])

## Reading the algebra

Two things to look at in a learned `A` (following the "PHNNs for finance"
notes): **asymmetry** -- $A_i \ne A_i^T$ says view $i$ modulates $j \to k$
differently from $k \to j$; and the **Gram matrix** $\langle A_i, A_j\rangle_F$
-- near-orthogonal $A_i$ are individually meaningful, collinear ones have
collapsed into each other (the gauge freedom of the Kronecker sum). Neither is
enforced here; they're measurements.

In [ ]:
#| export
def algebra_stats(A):
    """`A: [n, n, n]` -> dict of asymmetry per component and the normalized
    Gram matrix between components."""
    A = A.detach().float()
    fro = A.flatten(1).norm(dim=1).clamp_min(1e-8)
    asym = (A - A.transpose(1, 2)).flatten(1).norm(dim=1) / fro
    An = A.flatten(1) / fro[:, None]
    return {"asymmetry": asym.cpu(), "gram": (An @ An.T).cpu()}


def show_algebra(A, title="learned algebra", cmap="RdBu_r"):
    """One panel per `A_i`, symmetric colour scale, plus the Gram matrix."""
    import matplotlib.pyplot as plt
    A = A.detach().float().cpu()
    n = A.shape[0]
    st = algebra_stats(A)
    fig, axes = plt.subplots(1, n + 1, figsize=(2.6 * (n + 1), 2.8))
    v = A.abs().max().item()
    for i, ax in enumerate(axes[:n]):
        ax.imshow(A[i].numpy(), cmap=cmap, vmin=-v, vmax=v)
        ax.set_title(f"A_{i}  asym {st['asymmetry'][i]:.2f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    im = axes[-1].imshow(st["gram"].numpy(), cmap=cmap, vmin=-1, vmax=1)
    axes[-1].set_title("Gram <A_i, A_j>", fontsize=9)
    axes[-1].set_xticks([]); axes[-1].set_yticks([])
    fig.colorbar(im, ax=axes[-1], shrink=0.8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.close(fig)
    return fig

In [ ]:
import matplotlib.pyplot as plt
st = algebra_stats(hamilton_basis())
assert st["asymmetry"][0] == 0 and torch.all(st["asymmetry"][1:] > 1.4), "1 is symmetric; i, j, k antisymmetric"
assert torch.allclose(st["gram"], torch.eye(4), atol=1e-6), "Hamilton basis is orthogonal"
fig = show_algebra(hamilton_basis(), "Hamilton")
assert isinstance(fig, plt.Figure)
fig

## Pipeline

On the real data, with the four orderings from `01_data` as the `n=4` views.
Two runs: raw log-prices (`f=1`) and time-scattering coefficients
(`f=n_paths`), each PH vs. real-conv baseline. `#| eval: false` -- needs the
`.pt` cache and a few minutes.

In [ ]:
#| eval: false
from shakeqr.data import *
from shakeqr.scattering import scattering, stack_views
opening, closing, tickers = get_nyse_data()
prices, returns, tk = prepare(opening, closing, tickers)
Zl = log_prices(prices)
returns_w = standardize(returns)
C_ret = build_corr(returns_w); D_ret = corr_to_dist(C_ret)
orders = {"greedy": greedy_order(C_ret), "spectral": spectral_order(C_ret),
          "tsp": two_opt(D_ret, greedy_edge_order(D_ret)), "hrp": hrp_order(D_ret)}
n = len(orders)

In [ ]:
#| eval: false
# 1. raw log-prices: [n, C, T] -> [T, n, C]
X_raw = views_to_batch(stack_views(Zl, list(orders.values())))
print("batch:", tuple(X_raw.shape))
ae_ph = make_ae(n, 1, h=16, z=4, hyper=True)
ae_re = make_ae(n, 1, h=16, z=4, hyper=False)
print(f"params  PH {count_params(ae_ph):,}   real {count_params(ae_re):,}")
h_ph = fit_ae(ae_ph, X_raw, epochs=40)
h_re = fit_ae(ae_re, X_raw, epochs=40)
print(f"final val  PH {h_ph['val'][-1]:.4f}   real {h_re['val'][-1]:.4f}")

In [ ]:
#| eval: false
show_algebra(ph_algebras(ae_ph)[0], "first layer, log-prices  (views: " + ", ".join(orders) + ")")

In [ ]:
#| eval: false
# 2. scattering features as the per-view feature axis: [n, C, P, T] -> [T, n*P, C]
S, paths = scattering(Zl, J=6, Q=1)
X_sc = views_to_batch(stack_views(S, list(orders.values())))
X_sc = (X_sc - X_sc.mean((0, 2), keepdim=True)) / (X_sc.std((0, 2), keepdim=True) + 1e-6)   # per-feature whitening
print("batch:", tuple(X_sc.shape))
ae_ph_sc = make_ae(n, len(paths), h=32, z=8, hyper=True)
ae_re_sc = make_ae(n, len(paths), h=32, z=8, hyper=False)
print(f"params  PH {count_params(ae_ph_sc):,}   real {count_params(ae_re_sc):,}")
h_ph_sc = fit_ae(ae_ph_sc, X_sc, epochs=40)
h_re_sc = fit_ae(ae_re_sc, X_sc, epochs=40)
print(f"final val  PH {h_ph_sc['val'][-1]:.4f}   real {h_re_sc['val'][-1]:.4f}")

In [ ]:
#| eval: false
show_algebra(ph_algebras(ae_ph_sc)[0], "first layer, scattering features")

In [ ]:
#| eval: false
# reconstruction of one held-out day, view 0 (greedy ordering), along the channel axis
import matplotlib.pyplot as plt
ae_ph.eval()
with torch.no_grad():
    d = -1
    xb = X_raw[d:d + 1]
    rec = ae_ph(xb)
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(xb[0, 0].cpu(), lw=0.6, label="data"); ax.plot(rec[0, 0].cpu(), lw=0.6, label="PH recon")
ax.set_xlabel("channel position (greedy order)"); ax.legend(); plt.tight_layout(); plt.close(fig); fig

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()